In [1]:
import torch
import torch.nn as nn
from torch.utils import data

In [2]:
dataset = []
f1 = open('data/train.data', 'r')
for i in f1:
    dataset.append(i)

In [3]:
test_data = []
f1 = open('trees/test.conll', 'r')
for i in f1:
    buff = i.split()
    if len(buff) != 0:
        test_data.append((" ".join(buff[:-1]), buff[-1]))

In [4]:
# Hyper Parameters
sequence_length = 32
input_size = 40
first_hidden_size = 200
second_hidden_size = 200
num_layers = 2
num_classes = 12
batch_size = 1000
num_epochs = 7
learning_rate = 0.01

In [5]:
import tensorflow as tf
train_dataset = tf.convert_to_tensor(dataset)
test_dataset = tf.convert_to_tensor(test_data)

/Users/lesleycordero/anaconda3/lib/python3.6/importlib/_bootstrap.py:219: RuntimeWarning: compiletime version 3.5 of module 'tensorflow.python.framework.fast_tensor_util' does not match runtime version 3.6
  return f(*args, **kwds)


In [6]:
# Data Loader (Input Pipeline)
train_loader = data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size, 
                                          shuffle=False)


In [7]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # Set initial states 
        h0 = tf.Variable(torch.zeros(self.num_layers, len(x), self.hidden_size)) 
        c0 = tf.Variable(torch.zeros(self.num_layers, len(x), self.hidden_size))
        
        # Forward propagate RNN
        out, _ = self.lstm(x, (h0, c0))  
        
        # Decode hidden state of last time step
        out = self.fc(out[:, -1, :])  
        return out

In [8]:
rnn = RNN(input_size, first_hidden_size, num_layers, num_classes)

In [9]:
# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(rnn.parameters(), lr=learning_rate)

In [10]:
# Train the Model
for epoch in range(num_epochs):
    for (points, labels) in test_data:
        points = tf.Variable(points)
        labels = tf.Variable(labels)
        
        # Forward + Backward + Optimize
        optimizer.zero_grad()
        outputs = rnn(points)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print ('Epoch [%d/%d], Step [%d/%d], Loss: %.4f' 
                   %(epoch+1, num_epochs, i+1, len(train_dataset)//batch_size, loss.data[0]))


TypeError: object of type 'Variable' has no len()